In [1]:
import os
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

# all data from GSE216037 is in supplementary files:

metadata_url = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE216037&format=file&file=GSE216037%5FGEO%2Dpdata%5F%2Ecsv%2Egz"
processed_data_url = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE216037&format=file&file=GSE216037%5FProcessed%5Fdata%2Ecsv%2Egz"

metadata_df = pd.read_csv(metadata_url, compression="gzip", encoding="latin-1")
metadata_df = metadata_df.drop(columns=['TNM'])
metadata_df = metadata_df.set_index("Sample")

df_processed_data = pd.read_csv(processed_data_url, compression="gzip")
df_processed_data = df_processed_data.rename(columns={"SYMBOL1": "Genes"})
df_processed_data = df_processed_data.set_index("Genes")
df_processed_data = df_processed_data.T
df_processed_data.index.name = "Sample"
df_processed_data.columns.name = None
 

In [2]:
# Join metadata and processed data on index (Sample)
data_file_name = f'{GEO_ID}_original.csv'
df_joined = metadata_df.join(df_processed_data, how='inner')
df_joined = df_joined.reset_index()
df_joined.to_csv(data_file_name, index=False)

df_joined

,Sample,TP53 Signture Score,TP53 Signture Group,TP53 Mutation,Overall survival(month),Survival state,Gender,Age,Tumor_size,Grade,...,LOC729998,GTF2IP12,LINC01668,TRBV6-3,DUX4L32,LBHD2,LINC02498,CENPVL2,MGC4859,HYMAI
0,BC1,0.645576,Low,TP53_WT,49.0,0,Man,unknown,unknown,unknown,...,191.241760,0.275613,1.070301,0.390085,0.000000,0.000000,0.0,0.000000,0.000000,0.006389
1,BC2,1.350214,High,TP53_WT,49.0,0,Man,Young,Small,HG,...,444.222931,0.328956,0.132734,0.000000,0.000000,0.000000,0.0,0.006221,0.000000,0.008353
2,BC3,0.466355,Low,TP53_WT,11.0,1,Female,Old,Small,HG,...,155.533737,0.337103,0.233640,0.000000,0.000000,0.238413,0.0,0.000000,0.000000,0.007309
3,BC4,0.315740,Low,TP53_WT,2.0,1,Man,Old,unknown,HG,...,182.693405,0.803473,0.181343,0.000000,0.000000,0.249737,0.0,0.045464,0.000000,0.008982
4,BC5,0.285227,Low,TP53_WT,27.0,1,Female,Young,Large,HG,...,305.762360,0.468516,0.480183,0.000000,0.000000,0.106599,0.0,0.000000,0.000000,0.051091
5,BC6,0.781187,High,TP53_WT,41.0,0,Man,Old,unknown,LG,...,383.056396,0.307291,4.166724,0.000000,0.000000,0.437799,0.0,0.000000,0.000000,0.005362
6,BC7,0.997928,High,TP53_WT,41.0,0,Man,Old,Large,HG,...,312.754364,0.723819,1.339518,0.000000,0.000000,0.333792,0.0,0.000000,0.000000,0.031792
7,BC8,-0.144136,Low,TP53_Mut,2.0,1,Female,Old,Large,unknown,...,544.618774,0.046168,0.000000,0.000000,0.000000,0.383863,0.0,0.000000,0.078197,0.169705
8,BC9,0.319691,Low,TP53_Mut,40.0,0,Man,Old,Large,HG,...,171.435959,0.452982,0.000000,0.946100,0.043825,0.194655,0.0,0.000000,0.000000,0.020011
9,BC10,0.761222,High,TP53_WT,23.0,1,Man,Old,unknown,unknown,...,197.131592,0.464120,3.750278,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.005258


# Generate description

In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)
    print("Description generated")

desc_data['data_summary'] = {
    'samples': metadata_df.reset_index().shape[0],
    'clinical_features': metadata_df.reset_index().shape[1],
    'genes': df_processed_data.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
